<hr/>

# [Generalized Linear Mixed Models](https://donnievin.github.io/ASDA_2/)
By: **Donovan Vincent Jr** - dvincen9@jh.edu <br/>
Adapted From: **Dr. Sergey Kushnarev's ASDA II** <br/>
Estimated Workthrough Time: **120 mins**

<hr/>

<h1><font color="blue">Introduction to GLMs</font></h1>

Topics: 
* Recalling Linear Models (LMs)
* Generalizing Linear Models to get GLMs
* Exponential Dispersion Family (EDFs)
* Linear Predictor, $\eta$
* Link functions, $g(\mu) = \eta$
* Expectation and Variances of EDFs
* Mean-Variance Relationship
* The likelihood equations

In [ ]:
import time
import scipy
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go

from IPython.display import clear_output


# <font color='lightblue'> Recalling Linear Models </font>

> What is a linear model?

A Linear Model (LM) is a statistcal equation that fits `parameters` ($\beta_i$) to model a relationship between `predictor` variables ($X_i$) and `response` variables ($Y_i$), even when the data is noisy with `errors` ($\epsilon_i$). In general, these equations take the form:

<div align="center">

$$
\mathbf{Y} = \mathbf{X} \mathbf{\beta} + \mathbf{\epsilon}
$$

</div>


--- 


> Error Constraints

There are 3 requirements for the noise term: 
1. <font color='red'>Centered at 0</font>: $\mathbb{E}[\epsilon]=0$ <br>
    a. This ensures $\mathbb{E}[Y] = \mathbb{E}[X \beta + \epsilon] = \mathbb{E}[X \beta] + \mathbb{E}[\epsilon] = \mathbb{E}[X \beta]$

2. <font color='red'>Uncorrelated</font> : $Cov(\epsilon_i , \epsilon_j)=0$ <br>
    a. This follows the assumption that each observation is an independent random sample
    b. Required for Gauss-Markov Theorem to the Ordinary Least Squares (OLS) estimate Unbiased (See ASDA I)

3. <font color='red'>Homoscedasticity (constant variance)</font>: $Var(\epsilon_i)= \sigma^2$ <br>
    a. This ensures $Var[Y] = \mathbb{E}[X \beta + \epsilon] = \mathbb{E}[X \beta] + \mathbb{E}[\epsilon] = \mathbb{E}[X \beta]$\
    b. Required for Gauss-Markov Theorem to ensure tht the Ordinary Least Squares (OLS) estimate is the Best Linear Unbiased Estimator (BLUE) (See ASDA I)

<font color='purple'> Note: Uncorrelated (no linear relationship) does not necessarily mean independent (joint distribution). Independence is a stronger condition </font>

--- 

> LMs for polynomials

When we call these LINEAR models, we are only referring to the powers of $\beta_i$. We are free to have anything we want in the design matrix, even powers. 

Example: 
* $y = \beta_0 + \beta_1 x_1 + \beta_2 x_1^2 + \beta_3 x_1^3 + ... + \beta_p x_1^p = \sum_{i=1}^p \beta_i x_1^i$
* $X = [1, x_1^2,  x_1^2 + x_1^3 + ... + x_1^p] \in \mathbb{R}^{nxp}$
* $y = X \beta$


In [ ]:
x1 = np.arange(0,10)
x2 = np.arange(10,20)

B0, B1, B2 = 4, 5, 6


# In numpy: loc=E[Y] ; scale = Std[Y] ; size = number of samples
y_ideal = (B2 * x2) + (B1 * x1) + B0
y_data = (B2 * x2) + (B1 * x1) + B0 + np.random.normal(loc=0, scale=20, size=len(x1))

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=x1, y=x2, z=y_ideal,
        mode='markers',
        name='Model: E[Y]'
    )
)

fig.add_trace(
    go.Scatter3d(
        x=x1, y=x2, z=y_data,
        mode='markers',
        name='Observed: XB + e'
    )
)


# Axis labels
fig.update_layout(
    scene=dict(
        xaxis=dict(title='Predictor 1', range=[0, 10]),
        yaxis=dict(title='Predictor 2', range=[10, 20]),
        zaxis=dict(title='Response', range=[60,200]),
        camera=dict(
            eye=dict(x=-0.8, y=1.5, z=0.1)
        )
    ),
    title='LMs: Model the relationship between noisy variables'
)

fig.show()


# <font color='lightblue'> Generalized Linear Models </font>

[MORE EXAMPLES](https://en.wikipedia.org/wiki/Exponential_family#cite_note-7)


> 3 main components needed to generalize a GLM 

1. <font color='red'>Random component </font>: Each $y_i \sim$ i.i.d. Exponential Dispeersion Family (EDF)
    * Think of EDFs as a general case wrapper that many common distributions can be rewritten in

2. <font color='red'>Linear Predictor </font>: $\implies \eta = X\beta$
    * $\eta$ = nx1 where $\eta_i = \sum_{j=1}^p X_{ij}\beta_j$
    * $X$ = nxp design (model) matrix
    * $\beta$ = px1 parameter column vector

3. <font color='red'>Link function </font>: Links the mean to the linear predictor $\implies g(\mu_i) = \eta_i$ 
    * Link function is the forward transformation ($g(\mu_i)$), response function is the backwards transformation ($g^{-1}(\eta_i)$)

---

> Useful Diagram 

<div align="center">

$$
\eta = \sum_{j=1}^p X_{ij}\beta_j
$$

$$
\mu_i = g^{-1}(\eta)
$$

$$
b'(\theta_i) = \mu_i
$$

$$
\beta_j \implies  \eta \implies \mu_i \implies \theta_i 
$$

$$
\beta_j \implies  \eta = \sum_{j=1}^p X_{ij}\beta_j \implies \mu_i = g^{-1}(\eta) \implies [b'(\theta_i)]^1(\mu_i) \implies \theta_i 
$$

</div>

---

# <font color='lightblue'> Exponential Dispersion Family (Random Component) </font>

We can model the probability density function (PDF) / probability mass functions (PMF) of a variety of distributions using a common form known as an exponential dispersion family: $Y_i \sim EDF(\theta_i, \phi_i)$:


<div align="center">

$$
f(y_i | \theta_i, \phi) = exp[ \frac{y_i \theta_i - b(\theta_i)}{a(\phi)} + c(y_i, \phi)]
$$

</div>


where:
* $\theta_i$ = Natural Parameter
* $b(\theta_i)$ = cumulant function
* $\phi_i$ = Dispersion parameter
* $c(y_i, \theta_i)$ = ??? (Normalizing constant)

## $y_i \sim Poisson(\mu_i)$


$f(y_i|\mu_i) =  \frac{\mu_i^{y_i} e^{-\mu_i}}{y_i!}$  Note: Taylor series expansion $e^\mu = \sum_{y=1}^\infty \frac{\mu^y}{y!}$


---

Rewrite

$f(y_i|\mu_i) =  \frac{\mu_i^{y_i}}{y_i!}  e^{-\mu_i}$

$f(y_i|\mu_i) =  e^{ln( \frac{\mu_i^{y_i}}{y_i!} )}  e^{-\mu_i}$

$f(y_i|\mu_i) =  e^{ln(\mu_i^{y_i}) - ln( {y_i!} )}  e^{-\mu_i}$

$f(y_i|\mu_i) =  e^{-\mu_i + y_i ln(\mu_i) - ln( {y_i!} )} $

---

Compare terms in the numerator with EDF

<font color='green'>Natural Parameter</font>: $y_i \theta_i = y_i ln(\mu_i) \implies$ $\theta_i = ln(\mu_i)$

<font color='green'>Cumulant Function</font>: $b(\theta_i) = -\mu_i \implies$ $b(\theta_i) = e^{\theta_i}$

<font color='green'>Dispersion Parameter</font>: $a(\phi) = 1$

<font color='green'>Normalizing Constant</font>: $c(y_i) = -ln(y_i!)$ 🐵

## $n_i y_i \sim Binomial(n_i, \pi_i)$

Typically Binomial distributions are $Y_i \sim Bin(n_i, \pi_i)$ where $Y_i$ is the number of successes. However, here it is more convenient to let $y_i$ be the proportion of successes in a sample of size $n_i$

$n_i y_i \sim Bin(\pi_i, n_i)$ 

$\mathbb{E}[y_i] = \pi_i$ ; $Var[y_i] = \frac{\pi_i (1 - \pi_i)}{n_i}$

$f(y_i|\pi_i, n_i) = \binom{n_i}{n_iy_i} \pi_i^{n_iy_i} (1-\pi_i)^{n_i - n_iy_i}$ 


---

Rewrite

$f(y_i|\pi_i, n_i) = \binom{n_i}{n_iy_i} \pi_i^{n_iy_i} (1-\pi_i)^{n_i - n_iy_i}$ 

$f(y_i|\pi_i, n_i) = \binom{n_i}{n_iy_i} (\frac{\pi_i}{1-\pi_i})^{n_iy_i} (1-\pi_i)^{n_i}$ 

$f(y_i|\pi_i, n_i) = e^{ ln[ \binom{n_i}{n_iy_i} (\frac{\pi_i}{1-\pi_i})^{n_iy_i} (1-\pi_i)^{n_i} ] }$ 

$f(y_i|\pi_i, n_i) = e^{ ln\binom{n_i}{n_iy_i} + n_iy_i ln(\frac{\pi_i}{1-\pi_i}) + n_i ln(1-\pi_i) }$ 


$f(y_i|\mu_i) =  e^{-\mu_i + y_i ln(\mu_i) - ln( {y_i!} )} $ 🐵

---

Compare terms in the numerator with EDF

<font color='green'>Natural Parameter</font>: $y_i \theta_i = (n_iy_i) ln(\frac{\pi_i}{1-\pi_i}) \implies$ $\theta_i = ln(\frac{\pi_i}{1-\pi_i})$

<font color='green'>Cumulant Function</font>: $b(\theta_i) = -ln(1-\pi_i) \implies b(\theta_i) = ln(1 + e^{\theta_i})$ (Solve for $\pi_i$ from Natural Parameter)

<font color='green'>Dispersion Parameter</font>: $a(\phi) = \frac{1}{n_i}$

<font color='green'>Normalizing Constant</font>: $c(y_i) = ln\binom{n_i}{n_iy_i}$ 🐵

`LOGISTIC REGRESSION AND LOG ODDS`

## Gamma

## Beta

## Chi-Squared

## $y_i \sim Normal(\mu_i, \sigma^2)$


$f(y_i|\mu_i, \sigma_i) =  \frac{1}{\sqrt{2 \pi \sigma^2}} e^{(-\frac{(y_i - \mu)^2}{2 \sigma^2})}$ 


---

Rewrite

$f(y_i|\mu_i, \sigma_i) =  \frac{1}{\sqrt{2 \pi \sigma^2}} e^{(-\frac{(y_i - \mu)^2}{2 \sigma^2})}$ 

$f(y_i|\mu_i, \sigma_i) =  e^{ln(\frac{1}{\sqrt{2 \pi \sigma^2}}) + (-\frac{(y_i - \mu)^2}{2 \sigma^2})}$ 


---

Compare terms in the numerator with EDF

<font color='green'>Natural Parameter</font>: $y_i \theta_i = y_i ln(\mu_i) \implies$ $\theta_i = \mu_i$

<font color='green'>Cumulant Function</font>: $b(\theta_i) = -\mu_i \implies$ $b(\theta_i) = \frac{\mu^2}{2}$

<font color='green'>Dispersion Parameter</font>: $a(\phi) = \sigma^2$

<font color='green'>Normalizing Constant</font>: $c(y_i) = ??? $ 🐵

# <font color='lightblue'> Linear Predictor (Systematic Component) </font>

$\eta_i = \sum_{j=1}^p X_{ij}\beta_j$

# <font color='lightblue'> Link Functions </font>

> Set up 
g(.) is a monotonic, differentiable function such that 

<div align="center">

$$
g(\mu_i) = \eta_i = \sum_{j=1}^p X_{ij}\beta_j
$$

</div>

--- 

> Canonical Link Function 

This is the $g$ that transforms $\mu_i$ into the natural parameter $\theta_i$. This makes it such that $g^{-1} = b'(\theta_i)$ and the Useful Diagram becomes: 

<div align="center">

$$
\beta_j \implies  \eta = \theta_i \implies \mu_i
$$

</div>

--- 

> Example Binomial 

From the GLM view, logistic regression becomes: 
<div align="center">

$$
ln(\frac{\mu_i}{1 - \mu_i}) = \eta_i = \sum_{j=1}^p X_{ij} \beta_j
$$

</div>


# <font color='lightblue'> Expectations and Variances of EDFs </font>

> Likelihood

While probability predicts outcomes from a model, the likelihood measures how well a model fits observed data. Since each data point is assumed to be a independent random observation, we can get the joint likelihood of observing all of these points (which determines the models learned probability function) by multiplying the individual probabilities for each datapoint.

<div align="center">

$$
\mathcal{l} = Pr(y_1, ... , y_n ) = Pr(y_1) Pr(y_2) Pr(y_3) ...  Pr(y_n) = \Pi_{i=1}^n f(y_i | \phi_i, \theta_i)
$$

</div>

---

> Log likelihood 

For numerical stability, we take the log of this function which turns the product into sums. This transformation is ideal because the log is a <font color='red'>monotonically increasing function </font>, ensuring that maximizing $log(\mathcal{l})$ is analogous to maximizing $\mathcal{l}$ when we "undo the transformation". 

<div align="center">

$$
\mathcal{L} = log(\mathcal{l}) = \sum_{i=1}^n f(y_i | \phi_i, \theta_i)
$$

</div>

---



> Fact 1: The expectation of the score function is 0. 


The <font color='red'>score function </font> is the first derivative (gradient) of the log likelihood, with respect to the natural parameter. Below is a short proof of why its 0:

<div align="center">

$$
\mathbb{E}[\frac{d \mathcal{L}}{d \theta}] = \mathbb{E}[\frac{d}{d \theta} ln(\mathcal{l})] = \mathbb{E}[\frac{1}{\mathcal{l}} \frac{d \mathcal{l}}{d \theta}] = \frac{1}{\mathcal{l}}  \mathbb{E}[\frac{d \mathcal{l}}{d \theta}] = \frac{1}{\mathcal{l}}  \int_y \mathcal{l} \frac{d \mathcal{l}}{d \theta} = \int_y \frac{d \mathcal{l}}{d \theta} = \frac{d}{d \theta} [\int_y \mathcal{l}] = \frac{d}{d \theta} 1 = 0
$$

</div>


Remember for derivation: 
* <font color='red'>Log likelihood definition </font>: $\mathcal{L} = ln(f(y_i | \theta_i, \phi))$
* <font color='red'>Chain Rule Derivative of ln function </font>:$\frac{d}{dx} ln(f(x)) = \frac{1}{f(x)} \frac{d}{dx}[f(x)]$
* <font color='red'> Expectation Definition </font> $\mathbb{E}[x]= \int x p(x) dx$
* <font color='red'>Fundamental Theorem of Calc</font>: $\frac{d}{dx} [\int f(x)dx] = \int [\frac{d}{dx} f(x)] dx$


<div align="center">

$$
\mathbb{E}[\frac{d \mathcal{L}}{d \theta}] = 0
$$

</div>


---


> Property 2: The negative Hessian equals the variance


<div align="center">

$$
-\mathbb{E}[\frac{d^2 \mathcal{L}}{d \theta^2}] = \mathbb{E}[(\frac{d \mathcal{L}}{d \theta})^2]
$$

</div>


--- 


> Connections


Using the above two facts, we get: 



<div align="center">

$$
\mathbb{E}[\frac{d \mathcal{L}}{d \theta}] = \mathbb{E}[\frac{Y_i - b'(\theta_i)}{a (\phi)}] = \frac{1}{a(\phi)} \mathbb{E}[Y_i - b'(\theta_i)] = \frac{1}{a(\phi)} ( \mathbb{E}[Y_i] - \mathbb{E}[b'(\theta_i)] ) \implies \mathbb{E}[Y_i] = \mathbb{E}[b'(\theta_i)]
$$

$$
\mathbb{E}[(\frac{Y_i - b'(\theta_i)}{a(\phi)})^2] = \frac{1}{a(\phi)^2} \mathbb{E}[(Y_i - b'(\theta_i))^2] = \frac{1}{a(\phi)^2} \mathbb{E}[(Y_i -\mu)^2] = \frac{1}{a(\phi)^2} Var(Y_i) \implies \frac{b''(\theta_i)}{a(\phi)} =  \frac{1}{a(\phi)^2} Var(Y_i)  \implies Var(Y_i) = a(\phi) b''(\theta_i)
$$
</div>


Thus we can conclude: 


<div align="center">

$$
\mathbb{E}[Y_i] = b' (\theta_i) = \mu_i
$$

$$
Var[Y_i] = a(\phi) b'' (\theta_i)
$$
</div>


Isn't this beautiful, it turns out that once we have the EDF form of our PDF, we can easily extract the mean (derivative of the cumulant function) and variance (second derivative times dispersion) just by looking at the natural parameter and dispersion parameter.



# <font color='lightblue'> The Likelihood Equations </font>

> Dependencies 

Our goal is to pick the parameters $\beta$ that will maximize the likelihood. However, we do so many transformations to get from $\beta$ to $\mathcal{L}$ [Remember $\mathcal{L}(\theta)$, $\theta(\mu)$, $\mu(\eta)$ and $\eta(\beta)$] so we need the chain rule:

<div align="center">

$$
\mathcal{L}(\theta(\mu(\eta(\beta))))
$$

$$
\frac{d \mathcal{L}}{d\beta} = \frac{d \mathcal{L}}{d\theta} \frac{d \theta}{d\mu} \frac{d \mu}{d\eta} \frac{d \eta}{d\beta}
$$

</div>

Earlier, we saw that
* $\frac{d \mathcal{L}}{d\theta}= \frac{y_i - b'(\theta_i)}{a(\phi)}$
* $\frac{d \theta}{d\mu}= [\frac{d\mu}{d\theta}]^{-1} = [\frac{d}{d\theta} b'(\theta)]^{-1} = [b''(\theta)]^{-1} = \frac{a(\phi)}{Var(y_i)}$
* $\frac{d \eta}{d\beta}= X_{ij}$


So we get the following relationship

<div align="center">

$$
\max_{\beta} \frac{d \mathcal{L}}{d\beta} = \max_{\beta} [ (\frac{y_i - b'(\theta_i)}{a(\phi)}) (\frac{a(\phi)}{Var(y_i)}) (\frac{d \mu}{d\eta}) (X_{ij}) ]
$$

$$
0 = \frac{X_{ij} (y_i - b'(\theta_i))}{Var(y_i)} \frac{d \mu}{d\eta}
$$

$$
0 = \frac{X_{ij} (y_i - \mu_i)}{Var(y_i)} \frac{d \mu}{d\eta}
$$

</div>

<font color='purple'> NOTE, Link function dependency is hidden in</font> $\frac{d \mu}{d\eta}$

---

> Connections

In matrix form, this becomes: 

<div align="center">

$$
0 = X^T D V^{-1} (Y - \mu)
$$

</div>

This should look extremely similar (actually identical) to the OLS solution

* $V = \sigma^2 I$
* $D = I$
* $\mu = X \beta$

<div align="center">

$$
0 = X^T I (\sigma^2 I)^{-1} (Y - X \beta)
$$

$$
0 =  X^T Y - (X^T X \beta)
$$

$$
(X^T X \beta) =  X^T Y
$$

$$
\beta = (X^TX)^{-1} X^TY
$$

</div>

# <font color='lightblue'> Mean-Variance Relationship </font>

> Theory

Notice, different distributions only appear in the likelihood equations via their mean, $\mu_i$, and variance, $Var[Y_i]$. This means that by looking at the relationship between the mean and variance, we can uniquely determine the EDF which should best model the data. 

--- 

>  Examples 

Here are a few examples of what we mean by $Var[Y_i] = \nu(\mu_i)$:
* <font color='red'> Normal</font>: $Var[Y_i] = \sigma^2$ (Constant function)
* <font color='red'> Poisson</font>: $Var[Y_i] = \mu_i$ (Identity function)
* <font color='red'> Binomial</font>: $Var[Y_i] = \frac{\mu_i (1-\mu_i)}{n_i}$ (Quadratic function)

# Further References

1. [CMU's Correlation vs Independent](https://www.stat.cmu.edu/~cshalizi/uADA/13/reminders/uncorrelated-vs-independent.pdf) Quick read and proof
2. [Evidently AI:](https://www.evidentlyai.com/ranking-metrics/ndcg-metric) NDCG@K Explained
3. [DigitalSreeni](https://www.youtube.com/watch?v=IMvunY3LrQI&t=406s) NDCG